In [1]:
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                              balanced_accuracy_score, roc_auc_score)

voice  = pd.read_csv('results_voice.csv')
spiral = pd.read_csv('results_spiral.csv')
mri    = pd.read_csv('results_mri.csv')
modality_data = {'Voice': voice, 'Spiral': spiral, 'MRI': mri}

N_REPEATS = 500
RNG = np.random.default_rng(42)
METRIC_NAMES = ['accuracy', 'f1', 'sensitivity', 'specificity', 'balanced_acc', 'auc']

def compute_metrics(y_true, y_pred, y_prob):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'sensitivity': recall_score(y_true, y_pred, zero_division=0),
        'specificity': recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        'balanced_acc': balanced_accuracy_score(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_prob),
    }

def single_modality(df):
    m = compute_metrics(df['label'].values, df['pred'].values, df['prob'].values)
    return {k: (v, 0.0) for k, v in m.items()}

def synthetic_fusion(mods, n_repeats=N_REPEATS):
    """Много раз случайно перемешиваем и обрезаем каждую модальность до размера
    самой маленькой когорты (отдельно по классу), пары составляем позиционно,
    вероятности усредняем поровну между модальностями."""
    per_repeat = {k: [] for k in METRIC_NAMES}
    for _ in range(n_repeats):
        fused_true, fused_prob = [], []
        for label in [0, 1]:
            pools = [modality_data[m][modality_data[m]['label'] == label]['prob'].values for m in mods]
            n = min(len(p) for p in pools)
            idxs = [RNG.choice(len(p), size=n, replace=False) for p in pools]
            probs_matrix = np.stack([pools[i][idxs[i]] for i in range(len(mods))], axis=1)
            fused_true.extend([label] * n)
            fused_prob.extend(probs_matrix.mean(axis=1))
        fused_true, fused_prob = np.array(fused_true), np.array(fused_prob)
        fused_pred = (fused_prob > 0.5).astype(int)
        m = compute_metrics(fused_true, fused_pred, fused_prob)
        for k in METRIC_NAMES:
            per_repeat[k].append(m[k])
    return {k: (np.mean(v), np.std(v)) for k, v in per_repeat.items()}

rows = []
for r in range(1, 4):
    for combo in combinations(['Voice', 'Spiral', 'MRI'], r):
        combo = list(combo)
        if len(combo) == 1:
            m, kind = single_modality(modality_data[combo[0]]), 'real (single modality)'
        else:
            m, kind = synthetic_fusion(combo), f'synthetic pairing (n_repeats={N_REPEATS})'
        row = {'combination': ' + '.join(combo), 'type': kind}
        row.update({k: (f"{mean:.3f} ± {std:.3f}" if std > 0 else f"{mean:.3f}") for k, (mean, std) in m.items()})
        rows.append(row)

table = pd.DataFrame(rows)
pd.set_option('display.width', 160)
print(table.to_string(index=False))
table.to_csv('ablation_fusion_table.csv', index=False)
print("\nСохранено: ablation_fusion_table.csv")

         combination                              type      accuracy            f1   sensitivity   specificity  balanced_acc           auc
               Voice            real (single modality)         0.781         0.837         0.750         0.875         0.812         0.797
              Spiral            real (single modality)         0.893         0.897         0.867         0.923         0.895         0.892
                 MRI            real (single modality)         0.575         0.622         0.700         0.450         0.575         0.634
      Voice + Spiral synthetic pairing (n_repeats=500) 0.815 ± 0.044 0.869 ± 0.031 0.945 ± 0.046 0.572 ± 0.096 0.758 ± 0.053 0.920 ± 0.037
         Voice + MRI synthetic pairing (n_repeats=500) 0.816 ± 0.026 0.880 ± 0.018 0.946 ± 0.033 0.490 ± 0.049 0.718 ± 0.029 0.805 ± 0.036
        Spiral + MRI synthetic pairing (n_repeats=500) 0.855 ± 0.035 0.858 ± 0.037 0.822 ± 0.056 0.894 ± 0.037 0.858 ± 0.034 0.881 ± 0.017
Voice + Spiral + MRI synthe